## Creating a Chat Application with MLflow and OpenAI API

![llm_chatbot_model_serving](./Assets/llm_chatbot_model_serving.png)

### Installing Utilities and Libraries

In [0]:
# Install the OpenAI and MLflow libraries.
# 'openai' is used to interact with OpenAI's language models.
# 'mlflow==3.0.1' is a specific version of MLflow, a platform for managing machine learning workflows.
%pip install openai mlflow==3.0.1

### Restarting our Python Environment

In [0]:
# Restart the Python environment to ensure all newly installed libraries are available.
# This is useful after installing packages with %pip so that the environment is refreshed.
dbutils.library.restartPython()

### Creating the OpenAI Client

In [0]:
from openai import OpenAI  # Import the OpenAI class from the openai library
import os

# Store the OpenAI API key in an environment variable for security
api_key = os.environ.get("OPENAI_API_KEY")

# Create an OpenAI client instance to interact with the OpenAI API
client = OpenAI(
    api_key = api_key,  # Use the API key from the environment variable
    base_url = "https://8259564857436390.0.gcp.databricks.com/serving-endpoints"  # The base URL for the Databricks model serving endpoint
)

### Sending a Simple Chat Request

In [0]:
# Send a chat completion request to the OpenAI API using the client instance
completion = client.chat.completions.create(
    model="databricks-llama-4-maverick",  # Specify the model to use for generating responses
    messages=[
        {
            "role":"system",  # Set the system role to define the assistant's behavior
            "content": [
                {
                    "type": "text", "text": "You are Batman, the protector of Gotham City"  # System prompt for the assistant
                }
            ]
        },
        {
            "role": "user",  # Set the user role to provide the user's input
            "content": [
                {
                    "type": "text", "text": "How's Gotham City doing today?"  # User's question to the assistant
                }
            ]
        }
    ]
)

# Display the generated response from the assistant
print(completion.choices[0].message.content)

### Generating a custom Python Function using MLflow that can be served as a Mosaic AI Endpoint

In [0]:
import mlflow  # Import the MLflow library for model management
from mlflow import pyfunc  # Import pyfunc module for custom Python models
import os

class BasicChatBot(pyfunc.PythonModel):  # Define a custom chatbot class inheriting from MLflow's PythonModel
    def __init__(self, model_name: str):  # Constructor to initialize the chatbot with a model name
        self.model_name = model_name  # Store the model name as an instance variable
    
    def chatCompletionsAPI(self, user_query):  # Method to send a chat request to the OpenAI API
        openai_client = OpenAI(  # Create an OpenAI client instance for API communication
            api_key = os.environ.get("OPENAI_API_KEY"),  # API key for authentication from environment variable
            base_url = "https://8259564857436390.0.gcp.databricks.com/serving-endpoints"  # Base URL for Databricks model serving
        )

        response = openai_client.chat.completions.create(  # Send a chat completion request to the model
            model = self.model_name,  # Specify the model to use
            messages = [  # List of messages to define the conversation
                {
                    "role": "system",  # System message to set assistant behavior
                    "content": [
                        {
                            "type": "text", "text": "You are Batman, the protector of Gotham City"  # System prompt
                        }
                    ]
                },
                {
                    "role": "user",  # User message containing the user's query
                    "content": [
                        {
                            "type": "text", "text": user_query  # Insert the user's question
                        }
                    ]
                }
            ],
            temperature=0.7  # Set the randomness of the model's response.
                            # Higher values (e.g., 0.8-1.0) make output more creative and diverse.
                            # Lower values (e.g., 0.1-0.3) make output more focused and deterministic.
                            # Adjusting temperature can help control the balance between creativity and reliability in responses.
        )

        return response.choices[0].message.content  # Return the generated response text
    
    def predict(self, context, data):  # Required method for MLflow pyfunc models to make predictions
        user_query = data["user_query"].iloc[0]  # Extract the user's query from the input DataFrame
        gpt_response = self.chatCompletionsAPI(user_query)  # Get the model's response using the chat API
        return gpt_response  # Return the response as the prediction

### Saving our Model

In [0]:
# Instantiate the BasicChatBot class with the specified model name.
# This creates a chatbot object that uses the "databricks-llama-4-maverick" model for generating responses.
# You can change the model name to use a different language model if desired.
test_model = BasicChatBot(model_name="databricks-llama-4-maverick") # e.g., "databricks-claude-sonnet-4-5"

In [0]:
from mlflow.models import infer_signature  # infer_signature captures input/output schema for reproducibility and validation
import pandas as pd  # pandas is used to create DataFrames for model input/output examples
import mlflow  # MLflow manages model lifecycle, including saving and loading models
import shutil  # shutil removes directories to avoid overwrite issues
import os  # os checks for directory existence

# MLflow's pyfunc interface allows you to package any Python model for deployment and inference.
# The 'signature' parameter documents the expected input and output schema, improving model reproducibility and validation.
# 'input_example' provides a sample input for UI, validation, and downstream consumers.
# The 'output_example' demonstrates the expected output format for consumers and UI preview.
# The 'infer_signature' function helps ensure that the model's input and output types are well-documented and validated.
# Removing the model directory before saving prevents errors from previous runs and ensures a clean save.

input_example = pd.DataFrame([
    {"user_query": "how is Gotham City today?"}  # Example user query to the chatbot
])

output_example = pd.DataFrame([
    {
        "predictions": "Gotham City is facing cloudy weather with rising tensions downtown."  # Example chatbot response
    }
])

# infer_signature inspects the input and output DataFrames to generate schema information for MLflow model tracking.
signature = infer_signature(input_example, output_example)

model_path = "basicchatbot"  # Directory where the model will be saved

if os.path.exists(model_path):
    shutil.rmtree(model_path)

mlflow.pyfunc.save_model(
    path=model_path,  # Directory to save the model
    python_model=test_model,  # The Python model object to be saved
    signature=signature,  # Documents input/output schema for validation and deployment
    input_example=input_example  # Provides a sample input for UI and validation
)

### Loading our Saved Model

In [0]:
# Load our custom chatbot model from the local directory where it was saved.
# mlflow.pyfunc.load_model loads the model so you can use it for predictions.
# 'model_path' is the directory containing the saved model files.
loaded_pyfunc_model = mlflow.pyfunc.load_model(model_path)

### Testing our Saved Model

In [0]:
# Create a pandas DataFrame containing a sample user query for the chatbot model
model_input = pd.DataFrame([{"user_query": "hello how are you?"}])

# Use the loaded MLflow pyfunc model to generate a response to the user query
model_response = loaded_pyfunc_model.predict(model_input)

# Print the chatbot's response to the console
print(model_response)

### Logging our Saved/Loaded Model

In [0]:
import mlflow  # Import MLflow, a library for managing machine learning models and experiments

run_id = None  # Initialize run_id to None; this will store the unique ID of the MLflow run

# Log the model as an artifact to MLflow tracking server
with mlflow.start_run() as run:  # Start a new MLflow run to track this experiment and its artifacts
    # Log the entire local model directory (model_path) as an artifact.
    # 'local_dir' specifies the path to the local directory containing the saved model files.
    # 'artifact_path' specifies the destination folder name under the run's artifact root in MLflow.
    # This allows the model to be versioned, shared, and later registered or deployed from MLflow.
    mlflow.log_artifacts(local_dir=model_path, artifact_path="BasicChatBot")  # Log the saved model directory as an artifact under the name "BasicChatBot"
    print(f"Model logged with run ID: {run.info.run_id}")  # Print the unique run ID for reference, useful for model registration or retrieval
    run_id = run.info.run_id  # Store the run ID for later use (e.g., model registration in Unity Catalog)

### Registering the Model in Unity Catalog from Run ID

In [0]:
# Register the logged model in Unity Catalog using MLflow's register_model function.
# The first argument specifies the source path to the model artifact using the run ID and artifact name.
# The second argument specifies the name under which the model will be registered in Unity Catalog.
mlflow.register_model(f"runs:/{run_id}/BasicChatBot", "BasicChatBot")

### Testing the Real-Time Endpoint

use the below sample input when testing your endpoint from the UI

In [0]:
{
  # The 'dataframe_split' key specifies the format for passing tabular data to the model endpoint.
  "dataframe_split": {
    # 'columns' lists the names of the columns in the input DataFrame.
    "columns": [
      "user_query"  # The column for user queries to the chatbot.
    ],
    # 'data' contains the actual data rows, each as a list of values corresponding to the columns.
    "data": [
      [
        "how is Gotham City doing today?"  # Example user query for the chatbot model.
      ]
    ]
  }
}